<a href="https://colab.research.google.com/github/chandraniraychowdhury5/DS/blob/main/Wind_Energy_FeatureSelection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler

from statsmodels.stats.outliers_influence import variance_inflation_factor

In [6]:
df = pd.read_csv('/content/drive/MyDrive/datasetAndProgram/WindDatasetNewFeatures.csv')
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
if 'Time' in df.columns:
    df = df.drop(columns=['Time'])
df.drop(columns=['WindSpeed_HubHeight_90', 'WindSpeed_HubHeight_70', 'WindSpeed_HubHeight_60', 'WindSpeed_HubHeight_50',
                 'WindSpeed_HubHeight_40', 'WindSpeed_HubHeight_30', 'WindSpeed_HubHeight_20', 'WindSpeed_HubHeight_10',
                 'Power_Lag_6', 'Power_Lag_2', 'Wind100_Lag_2', 'Wind100_Lag_6'], inplace=True)


df.head(2)

,temperature_2m,relativehumidity_2m,dewpoint_2m,windspeed_10m,windspeed_100m,winddirection_10m,winddirection_100m,windgusts_10m,Power,Wind_Shear_Coefficient,...,Wind_Veering,U10,V10,U100,V100,Turbulence_Intensity,Gust_Factor,Kinetic_Energy_Flux,Power_Lag_1,Wind100_Lag_1
6,26.7,92,24.9,2.1,5.82,65,78,4.7,0.0959,0.442704,...,13,1.903246,0.887498,5.692819,1.210046,0.317261,0.807560,128.547268,0.0778,5.59
7,28.4,91,26.1,3.1,6.24,69,78,4.8,0.1141,0.303823,...,9,2.894099,1.110941,6.103641,1.297369,0.339454,0.769231,157.868118,0.0959,5.82


In [7]:
#seperate target and features
target = 'Power'
x = df.drop(columns = [target])
y = df[target]
print("*" * 60)
print('feature shape: ', x.shape)
print('target shape: ', y.shape)
print("*" * 60)


************************************************************
feature shape:  (43791, 21)
target shape:  (43791,)
************************************************************


Correlation with Target

Measures the direct linear relationship between each individual input feature and the target variable.

In [8]:
corr_target = df.corr(method = 'spearman')[target].sort_values(ascending=False)
print("\nCorrelation with Power\n")
print(corr_target)


Correlation with Power

Power                     1.000000
Power_Lag_1               0.991845
Kinetic_Energy_Flux       0.798030
Wind100_Lag_1             0.797661
WindSpeed_HubHeight_80    0.796872
windspeed_100m            0.796201
windspeed_10m             0.739117
windgusts_10m             0.650810
Air_Density               0.212737
winddirection_100m        0.150872
winddirection_10m         0.137066
Wind_Shear_Coefficient    0.087632
Wind_Veering             -0.046835
relativehumidity_2m      -0.073993
Gust_Factor              -0.103590
V100                     -0.157114
V10                      -0.171265
temperature_2m           -0.210961
dewpoint_2m              -0.228506
U10                      -0.355779
U100                     -0.369433
Turbulence_Intensity     -0.472107
Name: Power, dtype: float64


Correlation Matrix

Computes Pearson or Spearman coefficients between all pairs of numerical variables to measure linear relationships.




In [9]:
corr_matrix = x.corr(method = 'spearman').abs()
print(corr_matrix)

                        temperature_2m  relativehumidity_2m  dewpoint_2m  \
temperature_2m                1.000000             0.113851     0.934375   
relativehumidity_2m           0.113851             1.000000     0.220301   
dewpoint_2m                   0.934375             0.220301     1.000000   
windspeed_10m                 0.114287             0.228205     0.189442   
windspeed_100m                0.176779             0.055450     0.189543   
winddirection_10m             0.176537             0.081140     0.196239   
winddirection_100m            0.179613             0.087794     0.200968   
windgusts_10m                 0.005607             0.361032     0.128292   
Wind_Shear_Coefficient        0.166273             0.471565     0.003219   
Air_Density                   0.999698             0.094124     0.941830   
WindSpeed_HubHeight_80        0.172161             0.072987     0.191162   
Wind_Veering                  0.215511             0.389527     0.075736   
U10         

Low Variance Feature Removal

Drops features whose variance falls below a user-defined threshold, meaning the values barely change across the dataset.



In [12]:
# Low Variance Feature Removal

selector = VarianceThreshold(threshold=0.01)
selector.fit(x)
selected_variance = x.columns[selector.get_support()]
print("\nRemaining after Variance Threshold")
print(selected_variance)


Remaining after Variance Threshold
Index(['temperature_2m', 'relativehumidity_2m', 'dewpoint_2m', 'windspeed_10m',
       'windspeed_100m', 'winddirection_10m', 'winddirection_100m',
       'windgusts_10m', 'WindSpeed_HubHeight_80', 'Wind_Veering', 'U10', 'V10',
       'U100', 'V100', 'Turbulence_Intensity', 'Gust_Factor',
       'Kinetic_Energy_Flux', 'Power_Lag_1', 'Wind100_Lag_1'],
      dtype='object')


Multicollinearity Detection (VIF)

Calculates the Variance Inflation Factor (VIF) for each variable by regressing it against all other features; a VIF>5 or 10 indicates severe multicollinearity.



In [13]:
#  Variance Inflation Factor (VIF)

scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)
vif = pd.DataFrame()
vif["Feature"] = x.columns

vif["VIF"] = [
    variance_inflation_factor(x_scaled, i)
    for i in range(x_scaled.shape[1])
]

vif = vif.sort_values("VIF", ascending=False)

print("\nVariance Inflation Factor\n")

print(vif)


Variance Inflation Factor

                   Feature            VIF
10  WindSpeed_HubHeight_80  114855.583870
4           windspeed_100m   93386.655825
9              Air_Density    1356.668446
3            windspeed_10m    1327.149238
0           temperature_2m    1257.788964
2              dewpoint_2m     316.418083
12                     U10      40.832505
14                    U100      38.862037
1      relativehumidity_2m      37.675682
15                    V100      34.839274
13                     V10      34.602113
7            windgusts_10m      18.575359
20           Wind100_Lag_1      15.867627
8   Wind_Shear_Coefficient       6.502094
5        winddirection_10m       5.781173
6       winddirection_100m       5.737021
18     Kinetic_Energy_Flux       5.571704
19             Power_Lag_1       2.785834
17             Gust_Factor       2.223541
16    Turbulence_Intensity       1.610508
11            Wind_Veering       1.119768


Mutual Information

Measures the amount of information shared between a feature and the target, capturing both linear and highly non-linear dependencies.



In [14]:
# Mutual Information

mi = mutual_info_regression(x, y)
mi = pd.Series(mi, index=x.columns)
mi = mi.sort_values(ascending=False)

print("\nMutual Information\n")

print(mi)


Mutual Information

Power_Lag_1               2.163729
Kinetic_Energy_Flux       0.531032
WindSpeed_HubHeight_80    0.524277
Wind100_Lag_1             0.524171
windspeed_100m            0.524164
windspeed_10m             0.405411
windgusts_10m             0.310004
U100                      0.226372
U10                       0.223758
V10                       0.223242
V100                      0.213988
Gust_Factor               0.151326
Wind_Shear_Coefficient    0.149856
Turbulence_Intensity      0.136983
Wind_Veering              0.104255
dewpoint_2m               0.069517
winddirection_100m        0.068221
winddirection_10m         0.063444
temperature_2m            0.063087
Air_Density               0.059752
relativehumidity_2m       0.025015
dtype: float64


Random Forest Feature Importance

Evaluates the mean decrease in impurity (Gini or variance) across all trees whenever a specific feature is used to split a node.



In [15]:
# Random Forest Feature Importance

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs= -1)

rf.fit(x, y)

importance = pd.Series(rf.feature_importances_, index=x.columns)

importance = importance.sort_values(ascending=False)

print("\nRandom Forest Importance\n")

print(importance)



Random Forest Importance

Power_Lag_1               0.985651
Turbulence_Intensity      0.001449
Gust_Factor               0.001171
V100                      0.000930
relativehumidity_2m       0.000911
Wind100_Lag_1             0.000876
Kinetic_Energy_Flux       0.000833
Wind_Shear_Coefficient    0.000757
U10                       0.000743
dewpoint_2m               0.000660
windspeed_100m            0.000655
V10                       0.000643
U100                      0.000633
windgusts_10m             0.000575
winddirection_10m         0.000569
windspeed_10m             0.000554
WindSpeed_HubHeight_80    0.000516
winddirection_100m        0.000515
Wind_Veering              0.000479
Air_Density               0.000456
temperature_2m            0.000424
dtype: float64


Recursive Feature Elimination (RFE)

Core Mechanism: Fits a model, ranks features by weight or importance, removes the weakest feature, and repeats the process until a specified number of features remain.

In [16]:
lr = LinearRegression()

rfe = RFE(estimator=lr, n_features_to_select=15)

rfe.fit(x, y)

rfe_features = x.columns[rfe.support_]

print("\nRFE Selected Features\n")

print(rfe_features)



RFE Selected Features

Index(['temperature_2m', 'windspeed_10m', 'windspeed_100m', 'windgusts_10m',
       'Wind_Shear_Coefficient', 'Air_Density', 'WindSpeed_HubHeight_80',
       'U10', 'V10', 'U100', 'V100', 'Turbulence_Intensity', 'Gust_Factor',
       'Power_Lag_1', 'Wind100_Lag_1'],
      dtype='object')


Lasso Feature Selection

Core Mechanism: Fits a linear model with an L1 regularisation penalty that forces the coefficients of less important, redundant features exactly to zero.

In [17]:
# LASSO Feature Selection

lasso = LassoCV(cv=5, random_state=42)

lasso.fit(x_scaled, y)

lasso_coef = pd.Series(lasso.coef_, index=x.columns)

lasso_selected = lasso_coef[lasso_coef != 0]

lasso_selected = lasso_selected.sort_values()

print("\nLASSO Selected Features\n")

print(lasso_selected)



LASSO Selected Features

Wind100_Lag_1          -0.007248
windgusts_10m          -0.004932
V100                   -0.003814
Kinetic_Energy_Flux    -0.001438
winddirection_10m      -0.000603
winddirection_100m     -0.000361
relativehumidity_2m     0.000299
Gust_Factor             0.000318
Air_Density             0.000894
Turbulence_Intensity    0.003145
U100                    0.003880
windspeed_100m          0.023847
Power_Lag_1             0.279493
dtype: float64
